In [38]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from lightgbm import LGBMRegressor


FILE_PATH = "banco de dados daniel.xlsx"
SHEET_INPUT = "Dataset sem TS"                # alterar para "Dataset com TS variável" 
TARGET_COL = "w (mm)"
RANDOM_STATE = 42
TEST_SIZE = 0.2
OUTPUT_PATH = "./predicoes_modelos_new_newsem_ts.xlsx"


In [39]:
def make_pipeline(model):
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    preprocessor = ColumnTransformer([
        ("num", num_pipe, []),  
        ("cat", cat_pipe, [])
    ], remainder="drop")
    pipeline = Pipeline([("pre", preprocessor), ("model", model)])
    return pipeline

def train_eval_on_df(df, model_ctor):
    df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)
    y = df[TARGET_COL].astype(float)
    X = df.drop(columns=[TARGET_COL])

    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

    model = model_ctor
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    preprocessor = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ], remainder="drop")
    pipeline = Pipeline([("pre", preprocessor), ("model", model)])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    return {
        "pipeline": pipeline,
        "X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test,
        "y_pred": y_pred,
        "metrics": {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2},
        "num_cols": num_cols,
        "cat_cols": cat_cols,
        "full_X": X,          
        "full_y": y
    }


In [40]:
df_input = pd.read_excel(FILE_PATH, sheet_name=SHEET_INPUT)

models = {
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE)
}
models["LGBM"] = LGBMRegressor(n_estimators=200, random_state=RANDOM_STATE)


print("Modelos a avaliar:", list(models.keys()))
print("Linhas no dataset de input:", len(df_input))


Modelos a avaliar: ['RandomForest', 'GradientBoosting', 'HistGradientBoosting', 'LGBM']
Linhas no dataset de input: 341


In [41]:
results = {}
for name, mdl in models.items():
    res = train_eval_on_df(df_input, mdl)
    results[name] = res
    m = res["metrics"]
    print(f"{name}: R2={m['R2']:.4f}  MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}")


# selecionar top 3 por R2
ranked = sorted(results.items(), key=lambda kv: kv[1]["metrics"]["R2"], reverse=True)
top3 = [name for name, _ in ranked[:3]]
print("Top 3 modelos selecionados:", top3)


RandomForest: R2=0.7202  MAE=0.0398  RMSE=0.0640
GradientBoosting: R2=0.8005  MAE=0.0360  RMSE=0.0540
HistGradientBoosting: R2=0.8554  MAE=0.0336  RMSE=0.0460
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001774 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 238
[LightGBM] [Info] Number of data points in the train set: 272, number of used features: 9
[LightGBM] [Info] Start training from score 0.261875
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

/home/wmpjrufg/Documents/ic_andre_rimes/env/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [42]:
predictions_df = pd.DataFrame(index=df_input.index)  
predictions_df["index"] = df_input.index  

for name in top3:
    info = results[name]
    num_cols = info["num_cols"]
    cat_cols = info["cat_cols"]
    model_obj = info["pipeline"].named_steps["model"]

    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                         ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    preprocessor = ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)], remainder="drop")
    pipeline_full = Pipeline([("pre", preprocessor), ("model", model_obj.__class__(**model_obj.get_params()))])

    X_full = info["full_X"]
    y_full = info["full_y"]

    pipeline_full.fit(X_full, y_full)
    preds_full = pipeline_full.predict(X_full)  

    predictions_df[name] = preds_full

predictions_df = predictions_df.drop(columns=["index"])
predictions_df.head()


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000226 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 268
[LightGBM] [Info] Number of data points in the train set: 341, number of used features: 9
[LightGBM] [Info] Start training from score 0.261672
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

/home/wmpjrufg/Documents/ic_andre_rimes/env/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,HistGradientBoosting,LGBM,GradientBoosting
0,0.157664,0.153309,0.158822
1,0.235033,0.230042,0.229018
2,0.244448,0.241737,0.254857
3,0.268570,0.278942,0.281044
4,0.275892,0.309596,0.342811


In [43]:
predictions_df.to_excel(OUTPUT_PATH, index=False)
print("Arquivo salvo em:", OUTPUT_PATH)
print("Colunas no arquivo:", list(predictions_df.columns))


Arquivo salvo em: ./predicoes_modelos_new_newsem_ts.xlsx
Colunas no arquivo: ['HistGradientBoosting', 'LGBM', 'GradientBoosting']
